In [ ]:
%pip install -q git+https://github.com/zhouy185/rosa-starter.git

In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

times = delivery_times("Far West", "Fri/Sat eve", 45, seed=1)
print(len(times), "orders; average delivery time", round(times.mean(), 1), "minutes")

210 orders; average delivery time 42.9 minutes


## Part I. Describe: How bad is it, really?

### (a) Percentage of late orders

An order is **late** when its delivery time is longer than the promised delivery time.

The function `late_percentage(zone, time_block, promise)` works as follows:

1. **Choose the zones and time blocks to include.** If `zone` is `'all'`, it uses every zone in `ZONES`; otherwise it uses only the zone given. `time_block` works the same way with `TIME_BLOCKS`. This one rule covers all four cases (one cell, a whole time block, a whole zone, or everything).
2. **Simulate each zone and time block pair.** It calls `delivery_times` with the given promise (and `seed=1`, so the results are the same every time the notebook runs).
3. **Pool the orders.** It adds up the number of orders and the number of late orders over all the pairs it included.
4. **Return the percentage:** late orders ÷ total orders × 100.

The orders are pooled instead of averaging each pair's percentage because the pairs have different numbers of orders. A busy pair should count for more than a quiet one, so the combined figure is the true share of all orders that arrived late.

In [3]:
def late_percentage(zone, time_block, promise=PROMISE, seed=1):
    """Percentage of orders delivered later than the promise.

    zone and time_block can each be a specific name or 'all'.
    """
    zones = ZONES if zone == 'all' else [zone]
    blocks = TIME_BLOCKS if time_block == 'all' else [time_block]

    total_orders = 0
    late_orders = 0
    for z in zones:
        for b in blocks:
            times = delivery_times(z, b, promise, seed=seed)
            total_orders += len(times)
            late_orders += np.sum(times > promise)

    return 100 * late_orders / total_orders

**Testing the function** with today's 45-minute promise, using one example of each of the four cases:

| Case | Zone | Time block |
|---|---|---|
| Specific zone and time block | `"Far West"` | `"Fri/Sat eve"` |
| All zones, one time block | `'all'` | `"Lunch"` |
| One zone, all time blocks | `"Central"` | `'all'` |
| Everything | `'all'` | `'all'` |

In [4]:
test_cases = [
    ("Far West", "Fri/Sat eve"),
    ('all', "Lunch"),
    ("Central", 'all'),
    ('all', 'all'),
]

for zone, block in test_cases:
    pct = late_percentage(zone, block, 45)
    print(f"Zone: {zone:<9} | Time block: {block:<12} | Late: {pct:.1f}%")

Zone: Far West  | Time block: Fri/Sat eve  | Late: 36.7%
Zone: all       | Time block: Lunch        | Late: 3.3%
Zone: Central   | Time block: all          | Late: 4.2%
Zone: all       | Time block: all          | Late: 8.3%


**What the output shows:** Overall, about 8% of Rosa's orders arrive late. The problem is far from evenly spread. In the Far West on Friday/Saturday evenings, more than a third of orders (about 37%) miss the 45-minute promise. Lunch across all zones (about 3%) and the Central zone across all time blocks (about 4%) are well below the overall rate. This matches what we saw in class: the lateness is concentrated in the Far West on weekends.

### (b) Ranking zone and time block pairs by late rate

Now we want to know **which** zone and time block pairs cause the most late orders under today's 45-minute promise.

Steps:
1. Loop over every zone in `ZONES` and every time block in `TIME_BLOCKS` (3 × 4 = 12 pairs).
2. For each pair, use `late_percentage` from part (a) to get its late rate at a 45-minute promise.
3. Store each result as a `(zone, time_block, late_rate)` tuple in a list.
4. Sort the list by late rate from highest to lowest (`sorted` with `key` on the rate and `reverse=True`) and print it with a rank number.

In [5]:
pair_rates = []
for zone in ZONES:
    for block in TIME_BLOCKS:
        rate = late_percentage(zone, block, PROMISE)
        pair_rates.append((zone, block, rate))

ranked = sorted(pair_rates, key=lambda pair: pair[2], reverse=True)

print(f"{'Rank':<5} {'Zone':<9} {'Time block':<12} Late rate")
for rank, (zone, block, rate) in enumerate(ranked, start=1):
    print(f"{rank:<5} {zone:<9} {block:<12} {rate:5.1f}%")

Rank  Zone      Time block   Late rate
1     Far West  Fri/Sat eve   36.7%
2     North     Fri/Sat eve   16.7%
3     Far West  Weekday eve   11.2%
4     Central   Fri/Sat eve    6.8%
5     North     Weekday eve    6.0%
6     Far West  Other          4.7%
7     Central   Weekday eve    4.2%
8     North     Lunch          3.7%
9     Central   Lunch          3.5%
10    Central   Other          2.6%
11    North     Other          2.5%
12    Far West  Lunch          2.5%


**What the output shows:** Friday/Saturday evening is the worst time block in every zone. **Far West on Fri/Sat evenings is by far the worst pair**: about 37% of its orders are late, more than double the next pair (North, Fri/Sat eve, about 17%). Far West on weekday evenings comes third at about 11%. Every other pair stays below 7%, and the quietest pairs (lunch and "Other" times) are only around 2.5–3.5%. So the late-delivery problem is concentrated in a few busy evening pairs rather than spread across the whole business. This suggests that one fixed promise for every zone and time block may not be the best choice.

### (c) Average delivery time

The function `average_delivery_time(zone, time_block, promise)` has the same structure as `late_percentage` in part (a), so it handles the same four cases:

1. **Choose the zones and time blocks to include.** `'all'` means every zone in `ZONES` (or every time block in `TIME_BLOCKS`); otherwise only the one given is used.
2. **Simulate each zone and time block pair** with `delivery_times` at the given promise and `seed=1`.
3. **Pool the orders.** It adds up the delivery times of all orders and counts the orders over every pair included.
4. **Return the average:** total delivery minutes ÷ total orders.

As in part (a), the orders are pooled rather than averaging each pair's average. This way pairs with more orders count for more, and the result is the true average over all of Rosa's orders.

In [6]:
def average_delivery_time(zone, time_block, promise=PROMISE, seed=1):
    """Average delivery time in minutes over all included orders.

    zone and time_block can each be a specific name or 'all'.
    """
    zones = ZONES if zone == 'all' else [zone]
    blocks = TIME_BLOCKS if time_block == 'all' else [time_block]

    total_minutes = 0
    total_orders = 0
    for z in zones:
        for b in blocks:
            times = delivery_times(z, b, promise, seed=seed)
            total_minutes += np.sum(times)
            total_orders += len(times)

    return total_minutes / total_orders

**Testing the function** with the 45-minute promise, using the same four example cases as part (a):

In [7]:
for zone, block in test_cases:
    avg = average_delivery_time(zone, block, 45)
    print(f"Zone: {zone:<9} | Time block: {block:<12} | Average: {avg:.1f} minutes")

Zone: Far West  | Time block: Fri/Sat eve  | Average: 42.9 minutes
Zone: all       | Time block: Lunch        | Average: 28.2 minutes
Zone: Central   | Time block: all          | Average: 28.5 minutes
Zone: all       | Time block: all          | Average: 31.2 minutes


**What the output shows:** Across all orders, the average delivery takes about 31 minutes, well inside the 45-minute promise. Lunch across all zones and the Central zone across all time blocks both average about 28 minutes. Even the worst pair, Far West on Fri/Sat evenings, averages about 43 minutes, which is still *under* the promise. Yet part (a) showed that about 37% of its orders are late. An average can sit comfortably below the promise while a large share of individual orders miss it, so the average alone hides how often customers are let down. Part (e) comes back to this.

### (d) Ranking zone and time block pairs by average delivery time

This repeats the steps of part (b), but ranks the pairs by average delivery time instead of late rate:

1. Loop over all 12 zone and time block pairs.
2. For each pair, use `average_delivery_time` from part (c) at the 45-minute promise.
3. Store each result as a `(zone, time_block, average)` tuple.
4. Sort from the longest average to the shortest and print with a rank number.

In [8]:
pair_averages = []
for zone in ZONES:
    for block in TIME_BLOCKS:
        avg = average_delivery_time(zone, block, PROMISE)
        pair_averages.append((zone, block, avg))

ranked_avg = sorted(pair_averages, key=lambda pair: pair[2], reverse=True)

print(f"{'Rank':<5} {'Zone':<9} {'Time block':<12} Avg time")
for rank, (zone, block, avg) in enumerate(ranked_avg, start=1):
    print(f"{rank:<5} {zone:<9} {block:<12} {avg:5.1f} min")

Rank  Zone      Time block   Avg time
1     Far West  Fri/Sat eve   42.9 min
2     North     Fri/Sat eve   36.8 min
3     Far West  Weekday eve   35.0 min
4     Central   Fri/Sat eve   32.1 min
5     North     Weekday eve   31.4 min
6     Far West  Lunch         30.6 min
7     Far West  Other         30.0 min
8     Central   Weekday eve   29.0 min
9     North     Lunch         28.7 min
10    North     Other         27.7 min
11    Central   Lunch         26.8 min
12    Central   Other         26.0 min


**What the output shows:** The busy evening pairs again come out on top. Far West on Fri/Sat evenings has the longest average (about 43 minutes), followed by North Fri/Sat eve (about 37) and Far West weekday eve (about 35). The averages fall in a narrow band, though, from about 26 to 43 minutes, and **every pair's average is below the 45-minute promise**. Looking only at this table, Rosa might think the promise is being kept everywhere. Part (b) showed that it is not.

### (e) Which measure matters more for Rosa: late rate or average delivery time?

To compare the two measures directly, the next cell puts the rankings from (b) and (d) side by side. For each pair it shows the average time and its rank, and the late rate and its rank. Pairs are listed in the order of the average-time ranking from (d).

In [9]:
# Look up each pair's late rate and rank from part (b)
late_lookup = {}
for rank, (zone, block, rate) in enumerate(ranked, start=1):
    late_lookup[(zone, block)] = (rank, rate)

print(f"{'Zone':<9} {'Time block':<12} {'Avg time':>9} {'Rank':>5}   {'Late rate':>9} {'Rank':>5}")
for avg_rank, (zone, block, avg) in enumerate(ranked_avg, start=1):
    late_rank, rate = late_lookup[(zone, block)]
    print(f"{zone:<9} {block:<12} {avg:6.1f} min {avg_rank:>5}   {rate:8.1f}% {late_rank:>5}")

Zone      Time block    Avg time  Rank   Late rate  Rank
Far West  Fri/Sat eve    42.9 min     1       36.7%     1
North     Fri/Sat eve    36.8 min     2       16.7%     2
Far West  Weekday eve    35.0 min     3       11.2%     3
Central   Fri/Sat eve    32.1 min     4        6.8%     4
North     Weekday eve    31.4 min     5        6.0%     5
Far West  Lunch          30.6 min     6        2.5%    12
Far West  Other          30.0 min     7        4.7%     6
Central   Weekday eve    29.0 min     8        4.2%     7
North     Lunch          28.7 min     9        3.7%     8
North     Other          27.7 min    10        2.5%    11
Central   Lunch          26.8 min    11        3.5%     9
Central   Other          26.0 min    12        2.6%    10


**Answer: the late rate (b) is more relevant to Rosa's decisions.**

**1. Rosa's costs come from late orders, not from the average.** Rosa pays a refund on every order that misses the promise, and every late customer orders less in the future (the `COSTS` dictionary). An order that arrives in 44 minutes costs nothing extra; one that arrives in 46 minutes triggers a refund and lost future orders. So what drives cost is *how many orders cross the promise*, which is exactly what the late rate measures. The average delivery time does not appear anywhere in that cost.

**2. The average hides the problem.** All 12 pairs have an average below 45 minutes, so by the average alone the promise looks safe everywhere. Yet about 8% of all orders are late. The clearest example is **Far West on Fri/Sat evenings: its average is 42.9 minutes, inside the promise, but 36.7% of its orders are late.** North on Fri/Sat evenings averages 36.8 minutes, about 8 minutes of buffer, and still 1 in 6 orders is late. An average can look fine because many fast deliveries cancel out the slow ones, but each slow delivery is a disappointed customer and a refund.

**3. The late rate shows how serious each pair is; the average flattens it.** Averages only range from about 26 to 43 minutes (the worst pair is about 1.6× the best). Late rates range from 2.5% to 36.7% (about 15×). Far West Fri/Sat eve is only about 6 minutes slower on average than North Fri/Sat eve, but its late rate is more than double. The late rate makes clear where Rosa's money is being lost.

**4. The two rankings disagree in ways that would mislead Rosa.** The top five pairs are the same in both rankings, so the average does point roughly toward the busy evenings. Below that, the rankings diverge. **Far West at lunch ranks 6th by average time (30.6 minutes) but 12th, the least late of all pairs, by late rate (2.5%).** Using the average, Rosa might spend effort fixing Far West lunches, which almost never break the promise.

**5. The decision Rosa faces in Part II is a promise, which is a threshold.** Changing the promise from 45 to, say, 50 minutes changes the late rate directly, and through it the refund and churn costs. The average delivery time cannot tell her how many orders would fall on each side of a new promise.

The average is still useful as a general measure of how fast the kitchen and drivers are. But for deciding what promise to make, and for estimating what broken promises cost, Rosa should use the **late rate**.

## Part II. Exploring the best promise

### (a) Total cost per late order

A late order costs Rosa in two ways:

1. **Refund:** she refunds the customer `COSTS['refund']` dollars straight away.
2. **Churn:** the unhappy customer places `COSTS['churn_orders']` fewer orders in the future. Each lost order is `COSTS['margin']` dollars of profit she never earns.

So the total cost of one late order is:

$$\text{cost per late order} = \text{refund} + \text{churn orders} \times \text{margin per order}$$

The function takes a costs dictionary as input (instead of reading `COSTS` directly) so that the web app in Part III can pass in different figures.

In [10]:
def cost_per_late_order(costs):
    """Total cost of one late order: the refund plus the profit lost from future orders."""
    return costs['refund'] + costs['churn_orders'] * costs['margin']


print("COSTS:", COSTS)
print(f"Cost per late order: ${cost_per_late_order(COSTS):.2f}")

COSTS: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
Cost per late order: $26.20


**What the output shows:** Each late order costs Rosa **\$26.20**: the \$10 refund plus 1.8 lost future orders × \$9 margin = \$16.20 of lost profit. That is about **2.9 times** the \$9 she earns on an order. In other words, a single late order wipes out the profit of roughly three on-time orders. This is why late orders matter so much to the choice of promise.

### (b) Choosing the best promise

**Net profit** for a zone and time block at a given promise is the profit collected from customers less the cost of late orders:

$$\text{net profit} = \text{orders} \times \text{margin} - \text{late orders} \times \text{cost per late order}$$

The promise pulls this in two directions:

- **A shorter promise** attracts more orders (more margin), but more of them arrive late (more late costs).
- **A longer promise** makes late orders rare, but fewer customers order at all.

The best promise balances the two. Because the number of orders *and* the delivery times both change with the promise, we must call `delivery_times` again for every promise we try. We cannot reuse the 45-minute results.

The helper below works out the net profit for one promise. It returns the order count and the number of late orders too, so we can see where the profit comes from.

In [11]:
def net_profit(zone, time_block, promise, costs, seed=1):
    """Net profit for one zone, time block and promise.

    Returns (net_profit, number_of_orders, number_of_late_orders).
    """
    times = delivery_times(zone, time_block, promise, seed=seed)
    orders = len(times)
    late = int(np.sum(times > promise))
    profit = orders * costs['margin'] - late * cost_per_late_order(costs)
    return profit, orders, late

#### How to choose the range of promises

Before writing the search, we need to decide which promises to try. To see what a sensible range looks like, the next cell tries a very wide range, 5 to 120 minutes in steps of 5, for the worst pair from Part I: Far West on Fri/Sat evenings.

In [12]:
print(f"Far West, Fri/Sat eve (cost per late order = ${cost_per_late_order(COSTS):.2f})\n")
print(f"{'Promise':>7} {'Orders':>7} {'Late':>5} {'Late %':>7} {'Net profit':>11}")
for promise in range(5, 125, 5):
    profit, orders, late = net_profit("Far West", "Fri/Sat eve", promise, COSTS)
    late_pct = 100 * late / orders if orders > 0 else 0
    print(f"{promise:>7} {orders:>7} {late:>5} {late_pct:>6.1f}% {profit:>11.1f}")

Far West, Fri/Sat eve (cost per late order = $26.20)

Promise  Orders  Late  Late %  Net profit
      5     242   242  100.0%     -4162.4
     10     242   242  100.0%     -4162.4
     15     241   241  100.0%     -4145.2
     20     241   239   99.2%     -4092.8
     25     239   233   97.5%     -3953.6
     30     237   218   92.0%     -3578.6
     35     232   179   77.2%     -2601.8
     40     224   133   59.4%     -1468.6
     45     210    77   36.7%      -127.4
     50     188    33   17.6%       827.4
     55     158     8    5.1%      1212.4
     60     121     2    1.7%      1036.6
     65      84     0    0.0%       756.0
     70      54     0    0.0%       486.0
     75      32     0    0.0%       288.0
     80      18     0    0.0%       162.0
     85      10     0    0.0%        90.0
     90       6     0    0.0%        54.0
     95       3     0    0.0%        27.0
    100       2     0    0.0%        18.0
    105       1     0    0.0%         9.0
    110       0     0 

**What the wide scan shows, and how it guides the range:**

- **Too short is pointless.** Up to about 20 minutes, almost every order is late (e.g. 241 of 241 orders at 15 minutes). Net profit is deeply negative, around −\$4,000, because each late order costs \$26.20 and only earns \$9. No promise in this region could ever be the best, so the **lower end** of the range only needs to start where some orders begin arriving on time.
- **Too long is also pointless.** Beyond about 60–65 minutes, there are no late orders left to save, so stretching the promise only drives customers away. At 110 minutes nobody orders at all. The **upper end** only needs to go a bit past the point where late orders reach zero.
- **The best promise must lie strictly inside the range.** If the best promise is at either end of the list, the true best may lie beyond it, and the range must be extended. The example range in the assignment, 5 to 50 minutes, shows the danger. For this pair it would recommend **50 minutes**, the top of the list, but the scan above shows that **55 minutes** earns more (\$1,212 vs \$827). A range that is too narrow quietly gives the wrong answer.
- **Step size.** Steps of 5 minutes give promises that are easy to communicate ("45 minutes", "50 minutes") and keep the number of simulations small. A finer step could be used later around the best value if needed.

Based on this, we use **20 to 100 minutes in steps of 5** for every pair. It starts where on-time deliveries begin and ends well past the point where late orders and demand both vanish. The search function below also checks the edge rule: it warns if the best promise lands on either end of the list, which would mean the range should be widened.

In [13]:
def best_promise(zone, time_block, promises, costs, seed=1):
    """Try every promise in `promises` and return the one with the highest net profit.

    Returns (best_promise, best_net_profit, results), where results is a list of
    (promise, orders, late_orders, net_profit) tuples, one per promise tried.
    """
    promises = sorted(promises)
    results = []
    best_p = None
    best_profit = None
    for promise in promises:
        profit, orders, late = net_profit(zone, time_block, promise, costs, seed=seed)
        results.append((promise, orders, late, profit))
        if best_profit is None or profit > best_profit:
            best_p = promise
            best_profit = profit

    if len(promises) > 1 and best_p in (promises[0], promises[-1]):
        print(f"Warning: the best promise ({best_p} min) is at the edge of the range "
              f"{promises[0]}-{promises[-1]} min; consider widening the range.")

    return best_p, best_profit, results


PROMISES = list(range(20, 105, 5))
print("Promises to try:", PROMISES)

Promises to try: [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]


The function sorts the promises, then loops through them. For each one it uses `net_profit` and keeps the promise with the highest net profit so far. If two promises tie, the shorter one is kept, because a faster promise is more attractive to customers. It returns the best promise, its net profit, and the full list of results so the web app can display them.

Its inputs are the zone, the time block, the list of promises and the costs dictionary, as the assignment asks, so the Streamlit app in Part III can reuse it unchanged.

**Testing the function** on Far West, Fri/Sat eve, first with our chosen range and then with the assignment's example range of 5 to 50 minutes, to show why the range matters:

In [14]:
zone, block = "Far West", "Fri/Sat eve"

p, profit, results = best_promise(zone, block, PROMISES, COSTS)
print(f"{zone}, {block}: recommended promise = {p} minutes, net profit = ${profit:,.2f}")

profit_45, orders_45, late_45 = net_profit(zone, block, 45, COSTS)
print(f"For comparison, today's 45-minute promise: net profit = ${profit_45:,.2f} "
      f"({orders_45} orders, {late_45} late)")

print("\nWith the assignment's example range of 5 to 50 minutes:")
p_narrow, profit_narrow, _ = best_promise(zone, block, list(range(5, 55, 5)), COSTS)
print(f"Recommended promise = {p_narrow} minutes, net profit = ${profit_narrow:,.2f}")

Far West, Fri/Sat eve: recommended promise = 55 minutes, net profit = $1,212.40
For comparison, today's 45-minute promise: net profit = $-127.40 (210 orders, 77 late)

With the assignment's example range of 5 to 50 minutes:
Recommended promise = 50 minutes, net profit = $827.40


**What the output shows:** For Far West on Fri/Sat evenings, the best promise is **55 minutes**, with a net profit of about **\$1,212** over four weeks. Today's 45-minute promise **loses money** in this pair (about −\$127): 77 of its 210 orders are late, and at \$26.20 each, the late costs are larger than all the margin earned. Moving to 55 minutes loses some customers (158 orders instead of 210), but late orders fall from 77 to just 8, which turns a loss into Rosa's best result for this pair.

The narrow 5–50 range recommends 50 minutes and triggers the edge warning. This confirms that the range has to reach past the best value.

#### Best promise for every zone and time block

Rosa needs a promise for each pair, so we run the same search for all 12 pairs and compare each one with today's 45-minute promise.

In [15]:
total_today = 0
total_best = 0

print(f"{'Zone':<9} {'Time block':<12} {'Best':>5} {'Net profit':>11} {'At 45 min':>10} {'Gain':>9}")
for zone in ZONES:
    for block in TIME_BLOCKS:
        p, profit, _ = best_promise(zone, block, PROMISES, COSTS)
        profit_45, _, _ = net_profit(zone, block, 45, COSTS)
        total_best += profit
        total_today += profit_45
        print(f"{zone:<9} {block:<12} {p:>5} {profit:>11,.1f} {profit_45:>10,.1f} {profit - profit_45:>9,.1f}")

print(f"\nTotal net profit over four weeks: today (45 min everywhere) = ${total_today:,.2f}, "
      f"best promises = ${total_best:,.2f}")
print(f"Gain: ${total_best - total_today:,.2f} ({100 * (total_best - total_today) / total_today:.1f}%)")

Zone      Time block    Best  Net profit  At 45 min      Gain
Central   Lunch           45     2,104.2    2,104.2       0.0
Central   Weekday eve     45     2,603.2    2,603.2       0.0
Central   Fri/Sat eve     50     1,399.0    1,369.4      29.6
Central   Other           45     1,579.0    1,579.0       0.0
North     Lunch           45     1,526.6    1,526.6       0.0
North     Weekday eve     45     1,857.0    1,857.0       0.0
North     Fri/Sat eve     50       996.4      695.0     301.4
North     Other           45     1,335.2    1,335.2       0.0
Far West  Lunch           45     1,001.4    1,001.4       0.0
Far West  Weekday eve     50     1,210.8    1,032.2     178.6
Far West  Fri/Sat eve     55     1,212.4     -127.4   1,339.8
Far West  Other           45     1,492.2    1,492.2       0.0

Total net profit over four weeks: today (45 min everywhere) = $16,468.00, best promises = $18,317.40
Gain: $1,849.40 (11.2%)


**Recommendation for Rosa:**

- **Keep the 45-minute promise in 8 of the 12 pairs** (all lunches, all "Other" times, and weekday evenings in Central and North). There, 45 minutes is already the most profitable choice: few orders are late, and a longer promise would only lose customers.
- **Lengthen the promise in the busy evening pairs where lateness is high:**
  - **Far West, Fri/Sat eve: 55 minutes.** This is the biggest change, turning a loss of about \$127 into a profit of about \$1,212.
  - **North, Fri/Sat eve: 50 minutes** (about \$301 more).
  - **Far West, Weekday eve: 50 minutes** (about \$179 more).
  - **Central, Fri/Sat eve: 50 minutes** (about \$30 more, a small gain).
- **Overall**, setting the promise by zone and time block raises Rosa's four-week net profit from about **\$16,468 to \$18,317**, a gain of about **\$1,849 (11%)**, with no change to driver staffing.

These are exactly the pairs Part I flagged as having the highest late rates. The analysis moves from describing the problem (where orders are late) to a decision (what promise to make in each place). The figures come from one simulated four-week period (`seed=1`), so small differences, such as Central Fri/Sat eve's \$30 gain, should be treated with caution. The large gain in Far West on weekend evenings is clear.